# Прогноз USD/RUB на 1–7 дней

Бейзлайны и ML-поправки к случайному блужданию. Обучение заканчивается 31 декабря 2024 года. Модели и веса ансамбля выбираются на последовательных фолдах 2021–2024 годов. Итоговые метрики считаются на общих датах прогноза 2025–2026 годов.

Снимок в `datasets/` содержит признаки, доступные на 00:00 МСК, и цели на семь горизонтов. Для части исследовательских признаков использованы заданные лаги публикации. Код обучения, подбора и расчёта метрик находится в [ml_experiment.py](../src/usd_rub/ml_experiment.py).

In [ ]:
import json
from dataclasses import asdict
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import display
from threadpoolctl import threadpool_limits

from usd_rub.ml_dataset import feature_hit_rates, feature_panels, read_dataset
from usd_rub.ml_experiment import candidates, compare

cwd = Path.cwd()
root = next(path for path in (cwd, *cwd.parents) if (path / "config/ml_holdout.json").is_file())
config_path = root / "config/ml_holdout.json"
dataset_path = root / "datasets/usd_rub_2011_2026.csv.gz"
output = root / "data/reports/ml_holdout_50pct"

config = json.loads(config_path.read_text())
test_start = config["test_start"]
min_hit_rate = config["min_hit_rate"]

## Датасет

Hit rate — доля дат с заполненным значением в обучающей части. В снимке оставлены признаки с hit rate от 50%. Перед каждой подгонкой применяется тот же порог на её обучающем окне. Пропуск в ряду может отражать появление источника или отсутствие публикации на дату прогноза.

In [ ]:
dataset = read_dataset(dataset_path)
hit_rates = feature_hit_rates(dataset)
assert hit_rates.ge(min_hit_rate).all()

panels = feature_panels(dataset)
display(dataset.groupby("split", sort=False).agg(dates=("usd_rub", "size")))
display(pd.Series({mode: p.shape[1] for mode, p in panels.items()}, name="features").to_frame())
display(hit_rates.sort_values().head(10).rename("train_hit_rate").to_frame())
display(dataset[["split", "usd_rub", "target_return_1d", "target_return_7d"]].head())

## Подбор и тест

Ridge, CatBoost, Extra Trees и HistGradientBoosting прогнозируют относительную поправку к действующему курсу. Сравниваются разные окна истории, наборы признаков и способы моделировать горизонты. Победители семейств и случайное блуждание участвуют в подборе неотрицательных весов ансамбля. Критерий — среднее отношение RMSE к RMSE случайного блуждания по фолдам и горизонтам.

Следующая ячейка запускает все конфигурации, пересчитывает семь бейзлайнов, фиксирует параметры до оценки теста и сохраняет прогнозы. Последний шаг проверяет вклад групп признаков, повторяя обучение после их удаления.

In [ ]:
grid = pd.DataFrame([asdict(candidate) for candidate in candidates()])
display(grid.groupby(["model", "setup"]).size().unstack(fill_value=0))

In [ ]:
with threadpool_limits(limits=2):
    run = compare(config_path, root / "data", output, dataset_path=dataset_path)

assert run["test_start"] == test_start
print(f"Выбранная модель: {run['champion']}")
weights = pd.Series(run["ensemble_weights"], name="weight")
display(weights.sort_values(ascending=False).to_frame())

## Ошибки на тесте

RMSE приведён в рублях. Для MAPE, sMAPE и RMSPE используются проценты. В сводке RMSE нормируется на случайное блуждание по каждому горизонту перед усреднением.

In [ ]:
summary = pd.read_csv(output / "summary.csv", index_col="model")
metrics = pd.read_csv(output / "metrics.csv")
comparison = metrics.loc[metrics.model.isin(["rw", "ar1", "ensemble", run["champion"]])]
display(summary.round(4))
display(comparison.pivot(index="horizon", columns="model", values=["rmse", "mape_pct"]).round(4))

In [ ]:
ax = comparison.pivot(index="horizon", columns="model", values="rmse").plot(
    marker="o", ylabel="RMSE, ₽", xlabel="Горизонт, календарных дней"
)
ax.set_xticks(sorted(comparison.horizon.unique()))
plt.show()

## Вклад групп признаков

Положительное значение означает, что удаление группы увеличило средний RMSE относительно случайного блуждания. Вклады пересекающихся групп складывать нельзя.

In [ ]:
ablation = pd.read_csv(output / "ablation.csv")
display(ablation.pivot(index="group", columns="model", values="gain_from_group_pp").round(2))